# 1 | Template
---

In [ ]:
# ==============================================================================
# 5-Tages-Kurs: KI-Agenten - TAG 5 BASISSKELETT (v4.1.1)
# Multi-Agent Supervisor Pattern (LangChain 1.0+ & LangGraph 0.3.x)
# Fokus: Implementierung der Routing-Logik und der Worker-Agenten (TODOs)
# ==============================================================================

import operator
from typing import TypedDict, Annotated, List, Union
from langgraph.graph import StateGraph, START
from langgraph.checkpoint import MemorySaver
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, AIMessage
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langchain import hub
# Importiere LangChain-Komponenten für Worker Agent Logik (Tag 1/2 Wissen)
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# --- 1. SETUP UND TOOLS (MÜSSEN AN TAG 1-3 DEFIERT WORDEN SEIN) ---

# Funktion zur Initialisierung (Analog zu Tag 1, Block 1)
def init_chat_model(model_name: str = "gpt-4o", temperature: float = 0.0):
    """Initialisiert das Chat-Modell mit der gewünschten Konfiguration."""
    # Verwende ein leistungsstarkes Modell für Routing-Entscheidungen
    return ChatOpenAI(temperature=temperature, model=model_name)

# Globaler LLM
llm = init_chat_model()

# Beispiel-Tools (TODO: Durch tatsächliche Tools ersetzen, z.B. web_search, calculator)
# Die Worker Agents nutzen diese Tools.
ALL_TOOLS = []
# Beispiel:
# from your_tools_module import web_search, calculator
# ALL_TOOLS = [web_search, calculator]


# --- 2. STATE-DEFINITION (TYPEDDICT) ---

# Der Zustand des Graphen, geteilt zwischen allen Nodes.
class AgentState(TypedDict):
    """
    Repräsentiert den aktuellen Zustand des Multi-Agenten-Graphen.
    Messages: Nachrichtenverlauf (wird akkumuliert).
    next_agent: Vom Supervisor gewählter nächster Node.
    """
    messages: Annotated[List[BaseMessage], operator.add]
    next_agent: str

# --- 3. WORKER AGENT IMPLEMENTIERUNG (TEMPLATE/STUBS) ---

# Der Supervisor Agent muss ein definiertes Output-Schema zurückgeben,
# damit das Routing korrekt funktioniert.
SUPERVISOR_PROMPT = """
Du bist der Supervisor in einem Team von spezialisierten KI-Agenten.
Deine Aufgabe ist es, die letzte Benutzeranfrage (in messages) zu analysieren
und zu entscheiden, welcher Worker-Agent als Nächstes die Aufgabe bearbeiten soll.

Verfügbare Worker-Agenten:
- 'worker_a': Für Aufgabe A (z.B. Recherche, Datenerfassung).
- 'worker_b': Für Aufgabe B (z.B. Analyse, Texterstellung, Formatierung).
- 'END': Wenn die Aufgabe abgeschlossen ist oder keine weiteren Worker benötigt werden.

Wähle immer nur einen der drei Optionen.
"""

def call_supervisor(state: AgentState):
    """
    NODE: Wählt den nächsten Worker-Agenten basierend auf der letzten Nachricht.
    Implementiert das Routing mithilfe eines strukturierten Outputs.
    """
    print(f"\n--- SUPERVISOR NODE: Routing-Entscheidung treffen... ---")

    # TODO: Logik implementieren (Structured Output + Prompting)
    # Nutzen Sie with_structured_output (Tag 2 Wissen) mit einem Pydantic Schema,
    # das 'next_agent: Literal["worker_a", "worker_b", "END"]' definiert.

    # --- MVP DUMMY LOGIC (MUSS ERSETZT WERDEN) ---
    last_message = state["messages"][-1].content
    if "recherche" in last_message.lower():
        next_agent_choice = "worker_a"
    elif "analyse" in last_message.lower():
        next_agent_choice = "worker_b"
    else:
        # Nach dem ersten Schritt oder bei einfacher Frage END
        next_agent_choice = "END"

    print(f"--- SUPERVISOR ROUTET ZU: {next_agent_choice} ---")
    return {"next_agent": next_agent_choice}
    # ---------------------------------------------


def call_worker_a(state: AgentState):
    """
    NODE: Worker A führt seine spezialisierte Aufgabe aus.
    (TODO: Implementieren Sie hier den AgentExecutor mit Tools/Chains)
    """
    print(f"\n--- WORKER A NODE: Aufgabe bearbeiten... ---")

    # TODO: Agent-Logik für Worker A implementieren (AgentExecutor + Tools)
    # Nutze create_tool_calling_agent (Tag 1 Wissen)
    # Beispiel eines einfachen LLM-Calls (Muss durch vollen Agenten ersetzt werden)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Du bist Worker A und spezialisiert auf Recherche."),
        ("human", state["messages"][-1].content)
    ])
    chain = prompt | llm

    response = chain.invoke({"input": state["messages"][-1].content})

    # Der Output muss 'messages' akkumulieren
    return {"messages": [AIMessage(content=f"[WORKER A ERGEBNIS] {response.content}")]}


def call_worker_b(state: AgentState):
    """
    NODE: Worker B führt seine spezialisierte Aufgabe aus.
    (TODO: Implementieren Sie hier den AgentExecutor mit Tools/Chains)
    """
    print(f"\n--- WORKER B NODE: Aufgabe analysieren... ---")

    # TODO: Agent-Logik für Worker B implementieren (AgentExecutor + Tools)

    # Beispiel eines einfachen LLM-Calls
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Du bist Worker B und spezialisiert auf Analyse und Zusammenfassung."),
        ("human", state["messages"][-1].content)
    ])
    chain = prompt | llm

    response = chain.invoke({"input": state["messages"][-1].content})

    return {"messages": [AIMessage(content=f"[WORKER B ERGEBNIS] {response.content}")]}


# --- 4. GRAPH-AUFBAU UND KOMPILIERUNG (STRUKTUR IST FIX) ---

def route_supervisor(state: AgentState):
    """
    Conditional Edge: Routing-Logik.
    """
    next_agent = state["next_agent"]
    # Gibt den Namen des nächsten Nodes oder die Termination zurück
    return next_agent

# 4.1 Graph initialisieren
#
builder = StateGraph(AgentState)

# 4.2 Nodes hinzufügen (RunnableLambda macht Funktionen zu LangGraph Nodes)
builder.add_node("supervisor", RunnableLambda(call_supervisor))
builder.add_node("worker_a", RunnableLambda(call_worker_a))
builder.add_node("worker_b", RunnableLambda(call_worker_b))

# 4.3 Definition der Kanten (Edges): Feste Supervisor-Struktur
# Worker A -> Supervisor (nach Abschluss)
builder.add_edge("worker_a", "supervisor")
# Worker B -> Supervisor (nach Abschluss)
builder.add_edge("worker_b", "supervisor")

# 4.4 Definition der Conditional Edges (Routing)
# START -> Supervisor
builder.add_edge(START, "supervisor")

# Supervisor -> Worker oder Ende
builder.add_conditional_edges(
    "supervisor",
    route_supervisor,
    {
        # Schlüssel MUSS mit dem Rückgabewert von call_supervisor übereinstimmen
        "worker_a": "worker_a",
        "worker_b": "worker_b",
        "END": "__end__" # Terminierung (LangGraph Konvention)
    }
)

# 4.5 Graph kompilieren und Checkpointing aktivieren (Tag 4 Wissen)
# Der Checkpointer speichert den State für spätere Fortsetzung
app = builder.compile(checkpointer=MemorySaver())


# ==============================================================================
# --- 5. TEST-LAUF ---
# (Dieser Abschnitt muss in Colab ausgeführt werden)
# ==============================================================================

if __name__ == "__main__":
    # Test-Anfrage (wird im State gespeichert)
    initial_message = [HumanMessage(content="Ich brauche eine Recherche zu den Top 3 KI-Agenten Frameworks 2025 und eine anschließende Analyse ihrer Marktrelevanz.")]
    thread_id = "project-test-1"

    print(f"\n=========================================================")
    print(f" STARTE MULTI-AGENT PROJEKT (Thread ID: {thread_id})")
    print(f"=========================================================\n")

    # Starte den Graphen
    result = app.invoke(
        {"messages": initial_message},
        config={"configurable": {"thread_id": thread_id}}
    )

    print("\n=========================================================")
    print(" ENDERGEBNIS DES GRAPHS")
    print("=========================================================")
    # Gib die letzte vom Graph akkumulierte Nachricht aus
    final_response = result["messages"][-1]
    print(f"Letzte Nachricht:\n{final_response.content}")

    # 🚨 WICHTIG: LangSmith-Trace URL hier ausgeben!
    # print(f"\nLangSmith Trace URL: <Hier LangSmith Trace URL einfügen>")

# 2 | Anpassungen/Todos
---

## 2. Spezifische Templates (Anpassungen an TODOs)

Die Teilnehmer erhalten das Basisskelett, ergänzt durch die folgenden spezifischen **TODOs** für ihre gewählte Option.

---

### 🅰️ Option A: Research-Team 🧠

**Rolle des Supervisors:** Wählt zwischen Recherche (Tool-basiert) und Analyse/Antwort (LLM-basiert).

| Rolle | Node-Name | Spezialisierung | Tools (muss der Worker nutzen) |
| :--- | :--- | :--- | :--- |
| **Worker A** | `researcher` | Web-Recherche | `web_search` (Internet-Zugriff) |
| **Worker B** | `analyst` | Zusammenfassung, Strukturierung | **KEINE** (nur LLM-Fähigkeiten) |

#### Anpassungen im Basisskelett

* `call_worker_a` wird zu **`call_researcher`**.
* `call_worker_b` wird zu **`call_analyst`**.
* Routing-Logik (`route_supervisor`) muss auf `"researcher"` oder `"analyst"` routen.

#### Schlüssel-TODOs (Für Teilnehmende)

* **`call_researcher`**: Baue einen **AgentExecutor** mit dem `web_search`-Tool.
* **`call_analyst`**: Baue einen **LLM-Chain** zur strukturierten Antwortgenerierung (z.B. Markdown-Formatierung).
* **`call_supervisor`**: Nutze `with_structured_output` (Tag 2 Wissen!) zur Wahl zwischen `"researcher"`, `"analyst"` oder `"END"`.

---

### 🅱️ Option B: Content-Team 📝

**Rolle des Supervisors:** Wählt zwischen Texterstellung (Writer) und Prüfung/Verbesserung (Editor).

| Rolle | Node-Name | Spezialisierung | Tools (muss der Worker nutzen) |
| :--- | :--- | :--- | :--- |
| **Worker A** | `writer` | Entwurf/Erstellung von Content | **KEINE** |
| **Worker B** | `editor` | Grammatik-Check, Stil-Optimierung | `file_reader` (zur Prüfung vorhandener Entwürfe) |

#### Anpassungen im Basisskelett

* `call_worker_a` wird zu **`call_writer`**.
* `call_worker_b` wird zu **`call_editor`**.
* Routing-Logik (`route_supervisor`) muss auf `"writer"` oder `"editor"` routen.

#### Schlüssel-TODOs (Für Teilnehmende)

* **`call_writer`**: Baue eine **LCEL-Chain** für den ersten Entwurf (Fokus auf Kreativität).
* **`call_editor`**: Baue einen Agenten, der das **`file_reader`**-Tool nutzt, um den Output des Writers zu prüfen und zu verbessern.
* **`call_supervisor`**: Definiere Routing, um *zuerst* den Writer zu starten, dann den Editor und dann **`END`**.

---

### 🚹 Option C: Support-Team ❓

**Rolle des Supervisors:** Wählt zwischen einfacher FAQ-Antwort (RAG-basiert) und komplexer Problemlösung (Tool-basiert).

| Rolle | Node-Name | Spezialisierung | Tools (muss der Worker nutzen) |
| :--- | :--- | :--- | :--- |
| **Worker A** | `faq_agent` | Beantwortung von Standardfragen | **RAG-Tool** (Vector Store Retriever) |
| **Worker B** | `specialist` | Komplexe/rechnerische Anfragen | `calculator` oder komplexes Fach-Tool |

#### Anpassungen im Basisskelett

* `call_worker_a` wird zu **`call_faq_agent`**.
* `call_worker_b` wird zu **`call_specialist`**.
* Routing-Logik (`route_supervisor`) muss auf `"faq_agent"` oder `"specialist"` routen.

#### Schlüssel-TODOs (Für Teilnehmende)

* **`call_faq_agent`**: Baue einen Agenten, der das **RAG-Tool** (Tag 3 Wissen!) priorisiert.
* **`call_specialist`**: Baue einen Agenten, der das **`calculator`**-Tool nutzt.
* **`call_supervisor`**: Routing-Logik: Unterscheide zwischen **wissensbasierten (FAQ)** und **aktionsbasierten (Specialist)** Anfragen.

---

## 💡 Trainer-Steuerung (Template-Nutzung)

Diese Templates **reduzieren das Risiko** drastisch, indem sie den Fokus der Teilnehmer auf die *Logik* und nicht auf die *Struktur* lenken.

| Situation | Intervention | Effekt |
| :--- | :--- | :--- |
| **Team ist stuck (LangGraph)** | "Nutzt das Basisskelett. Ändert nur die **`TODO:`** Kommentare." | **Eliminierung von 70%** der LangGraph-Aufbaufehler. |
| **Team ist stuck (Logik)** | "Fokus auf **`call_supervisor`**. Der Supervisor muss nur **'researcher'** oder **'END'** zurückgeben können, um den **MVP** zu erfüllen." | **Scope-Begrenzung** auf das Wesentliche des Musters. |
| **Team will 3+ Worker** | "Bleibt beim Template, 2 Worker reichen. Das Ziel ist das **Muster** zu verstehen, nicht die Größe des Teams." | **Verhinderung** von Chaos und Zeitverlust. |